# Stage C 03j — freeze nested complete-replicon panels and c16 baseline


In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='df20b8614ca3165d365dd945948ca24d9495d27e'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
DATASET_NAME='nonoverlap_6mer_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
ACCESSION_MANIFEST=TAXONOMY_MANIFEST
ANI_MEMBERSHIP=f'{DRIVE_ROOT}/stage_c_dataset/manifests/ani99_membership.parquet'
ANI_PAIRS=f'{DRIVE_ROOT}/stage_c_dataset/manifests/skani_triangle.tsv'


In [ ]:
from pathlib import Path
from google.colab import drive
import json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/DATASET_NAME
panels=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3/panels'
PROTOCOL=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
def run_logged(root,label,command):
    root.mkdir(parents=True,exist_ok=True)
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(root),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (root/'FAILED.txt',root/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-20000:])
        raise


In [ ]:
for path in map(Path,(ACCESSION_MANIFEST,ANI_MEMBERSHIP,ANI_PAIRS)):
    if not path.is_file(): raise FileNotFoundError(path)
if not (panels/'panel_summary.json').is_file():
    run_logged(panels,'freeze_ecoli_panels',['seqtrainer-titans-stage-c-panel','freeze','--dataset-dir',str(dataset),'--accession-manifest',ACCESSION_MANIFEST,'--ani-membership',ANI_MEMBERSHIP,'--ani-pairs',ANI_PAIRS,'--output-dir',str(panels)])
for name in ('e25','e100','e250','e100_additions','validation','test'):
    subprocess.run(['seqtrainer-titans-stage-c-panel','validate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/f'{name}.json')],check=True)
print((panels/'panel_summary.json').read_text())


In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime for the c16 baseline.')
baseline=Path(DRIVE_ROOT)/'runs/c17_v3_c16_broad_baseline'
c16=Path(DRIVE_ROOT)/'runs/c16_deep_adaptive_5m_paper_exact/latest.pt'
run_logged(baseline,'evaluate_c16',['seqtrainer-titans-stage-c-evaluate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'validation.json'),'--run',f'c16={c16}','--output-dir',str(baseline/'evaluation'),'--split','val','--comparison-mode','partial','--device','cuda','--protocol',str(PROTOCOL),'--run-id','c16_broad_ecoli_baseline_v1'])
if not shutil.which('prodigal'):
    subprocess.run(['apt-get','update'],check=True); subprocess.run(['apt-get','install','-y','prodigal'],check=True)
run_logged(baseline,'generate_c16',['seqtrainer-titans-stage-c-generate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'validation.json'),'--taxonomy-manifest',TAXONOMY_MANIFEST,'--checkpoint',str(c16),'--output-dir',str(baseline/'generation_t0p6'),'--split','val','--species','Escherichia coli','--prompts','4','--prompt-tokens','128','--new-tokens','1024','--temperatures','0.6','--top-k','1024','--top-p','0.99','--device','cuda','--memory-mode','adaptive','--prodigal',shutil.which('prodigal'),'--protocol',str(PROTOCOL),'--run-id','c16_broad_ecoli_baseline_v1'])
print('SHARE THIS DIRECTORY:',baseline)
